SEGMENTATION

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

import cv2
from sklearn.model_selection import train_test_split
import albumentations as A

CHECK GPU

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device) #prints out "cuda" if a GPU is available, otherwise "cpu", it helps check if GPU is enabled

FILE PATHS FOR THE DATASETS

In [ ]:
from google.colab import drive
drive.mount('/content/drive') #mounts Google Drive to access the dataset stored there

IMAGE_DIR = "/content/drive/MyDrive/KidneyStone/images" #the original images are stored in this directory
MASK_DIR = "/content/drive/MyDrive/KidneyStone/masks" #the masks are stored in this directory

GETTING THE TOTAL COUNT OF THE DATASET (TO MAKE SURE EVERY THE TOTAL NUMBER OF IMAGE AND MASKS ARE CORRRESOPONDING)

In [ ]:
images = sorted(os.listdir(IMAGE_DIR)) #list all the files in the IMAGE_DIR and sort them in ascending order
masks = sorted(os.listdir(MASK_DIR))  #list all the files in the MASK_DIR and sort them in ascending order

print(len(images))
print(len(masks))

VISUALIZING ONE IMAGE SO IT CAN BE INSPECTED

In [ ]:
#Getting the path to the first image and mask in the directories
image_path = os.path.join(
    IMAGE_DIR,
    images[0]
)

mask_path = os.path.join(
    MASK_DIR,
    masks[0]
)
#Reading the image and mask using OpenCV, cv2.imread() is used to read the image and mask from the specified paths
image = cv2.imread(
    image_path,
    cv2.IMREAD_GRAYSCALE
) #

mask = cv2.imread(
    mask_path,
    cv2.IMREAD_GRAYSCALE
)

#working with plt to display the image and mask, plt is a plotting library in Python that is used to create visualizaton
plt.figure(figsize=(10,5)) #used to set the size of the figure or image to 10 inches wide and 5 inches tall  

plt.subplot(1,2,1) #used to create a subplot with 1 row and 2 columns, and the current plot will be the first one (1,2,1) means 1 row, 2 columns, and the first plot
plt.imshow(image,cmap="gray") #used to display the image in grayscale, cmap="gray" is used to specify that the image should be displayed in grayscale
plt.title("CT Image") #used to set the title of the plot to "CT Image"

plt.subplot(1,2,2) #same setting for the images, doing the same for the mask, but this time the current plot will be the second one (1,2,2) means 1 row, 2 columns, and the second plot
plt.imshow(mask,cmap="gray")
plt.title("Mask")

plt.show()

PAIRING EACH IMAGE WITH ITS MASK

In [ ]:
# getting all image names for the for loop
all_images = sorted(os.listdir(IMAGE_DIR)) #list all the files in the IMAGE_DIR and sort them in asceding order

pairs = [] #empty list to store the pairs of image and mask paths
for image_name in all_images:
    image_path = os.path.join(IMAGE_DIR, image_name) #get the path to the image by joining the IMAGE_DIR and the image name
    mask_path = os.path.join(MASK_DIR, image_name) #get the path to the mask by joining the MASK_DIR and the image name, assuming that the mask has the same name as the image

    if not os.path.exists(mask_path):
        print(f"Missing mask for: {image_name}")
        continue
    
    pairs.append((image_path, mask_path)) #append the tuple of image path and mask path to the pairs list   
print(len(pairs))

SPLITTING THE PAIRED DATASET

In [ ]:
train_pairs, temp_pairs = train_test_split(
    pairs,
    test_size=0.3, #Means that 30% of the dataset is reserved for validation and testing, while the remaining 70% is used for training
    random_state=42 #locking the shuffle so it never changes between runs.
)

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.5, #Means that 50% of the remaining dataset (which is 30% of the original dataset) is reserved for validation and the other 50% is reserved for testing
    random_state=42 #locking the shuffle so it never changes between runs.
)

print(len(train_pairs))
print(len(val_pairs))
print(len(test_pairs))

THE CRAZY PYTOUCH DATASET CLASS

In [ ]:
class KidneyDataset(Dataset):
    def __init__(self, pairs, transform = None, normalize ='imagenet'):
        self.pairs = pairs
        self.transform = transform
        self.mean = 0.485
        self.std = 0.229

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Error handling for corrupt files
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        if mask is None:
            raise ValueError(f"Failed to load mask: {mask_path}")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        image = image / 255.0 #scale to (0, 1)
        image = (image - self.mean) / self.std #imagenet standardization
        mask = mask / 255.0
        #convert to torch tensors
        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.float32)
        
        # Both image and mask: add channel dimension (1 channel for grayscale)
        image = image.unsqueeze(0)  # (1, H, W) for grayscale input
        mask = mask.unsqueeze(0)    # (1, H, W) for binary mask

        return image, mask

AUGMENTATION PIPELINE

In [ ]:
train_transform = A.Compose([
        A.Resize(256,256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),
        A.RandomBrightnessContrast( p=0.3)
    ])

val_transform = A.Compose([
        A.Resize(256,256)
    ])

LOAD DATASETS INTO PYTORCH CLASS

In [ ]:
train_dataset = KidneyDataset(train_pairs, train_transform)

val_dataset = KidneyDataset(val_pairs, val_transform)

test_dataset = KidneyDataset(test_pairs, val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

VIEW THE PROCESSED DATASET AND CREATE A BATCH

In [ ]:
image, mask = train_dataset[0]
batch_image, batch_mask = next(iter(train_loader))

print("Single Image Shapes:")
print(f"  Image: {image.shape} (expected: torch.Size([1, 256, 256]))")
print(f"  Mask: {mask.shape} (expected: torch.Size([1, 256, 256]))")

print("\nBatch Shapes:")
print(f"  Batch Image: {batch_image.shape} (expected: torch.Size([16, 1, 256, 256]))")
print(f"  Batch Mask: {batch_mask.shape} (expected: torch.Size([16, 1, 256, 256]))")

# Verify shapes are correct
assert image.shape == torch.Size([1, 256, 256]), f"Wrong image shape: {image.shape}"
assert mask.shape == torch.Size([1, 256, 256]), f"Wrong mask shape: {mask.shape}"
assert batch_image.shape == torch.Size([16, 1, 256, 256]), f"Wrong batch image shape: {batch_image.shape}"
assert batch_mask.shape == torch.Size([16, 1, 256, 256]), f"Wrong batch mask shape: {batch_mask.shape}"

print("\n All shapes are correct!")

LOAD PRETRAINED MODEL

In [ ]:
resnet = torchvision.models.resnet50(weights="DEFAULT")

ENCODER, DECODER AND FORWARD FUNCTION

In [ ]:
class ResUNet(nn.Module):

    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet50(weights="DEFAULT")

        # Adapt first conv layer to accept 1 channel input instead of 3
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.enc2 = resnet.layer1
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.enc5 = resnet.layer4

        # decoder with skip connections
        self.up1 = nn.ConvTranspose2d(2048, 1024, kernel_size=2, stride=2)
        self.conv1 = nn.Conv2d(2048, 512, 3, padding=1)  # 1024 (from up1) + 1024 (skip from enc4)
        
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.conv2 = nn.Conv2d(768, 256, 3, padding=1)   # 256 (from up2) + 512 (skip from enc3)
        
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.conv3 = nn.Conv2d(384, 128, 3, padding=1)   # 128 (from up3) + 256 (skip from enc2)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = nn.Conv2d(128, 64, 3, padding=1)    # 64 (from up4) + 64 (skip from enc1)
        
        self.up5 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.final = nn.Conv2d(32, 1, 1)


    def forward(self, x):
        # Encoder path with skip connections saved
        x1 = self.enc1(x)                      # 64 channels

        x2 = self.pool(x1)
        x2 = self.enc2(x2)                     # 256 channels

        x3 = self.enc3(x2)                     # 512 channels
        x4 = self.enc4(x3)                     # 1024 channels
        x5 = self.enc5(x4)                     # 2048 channels

        # Decoder path with skip connections
        x = self.up1(x5)                       # 1024 channels
        x = torch.cat([x, x4], dim=1)          # Concatenate with x4 (skip connection) → 2048
        x = torch.relu(self.conv1(x))          # 512 channels

        x = self.up2(x)                        # 256 channels
        x = torch.cat([x, x3], dim=1)          # Concatenate with x3 (skip connection) → 768
        x = torch.relu(self.conv2(x))          # 256 channels

        x = self.up3(x)                        # 128 channels
        x = torch.cat([x, x2], dim=1)          # Concatenate with x2 (skip connection) → 384
        x = torch.relu(self.conv3(x))          # 128 channels

        x = self.up4(x)                        # 64 channels
        x = torch.cat([x, x1], dim=1)          # Concatenate with x1 (skip connection) → 128
        x = torch.relu(self.conv4(x))          # 64 channels

        x = self.up5(x)                        # 32 channels
        x = self.final(x)                      # 1 channel
        
        return torch.sigmoid(x)

CREATE MODEL

In [ ]:
model = ResUNet()
model = model.to(device)

print(model)

DICE LOSS

In [ ]:
def dice_loss(pred, target):
    smooth = 1e-6
    intersection = (pred * target).sum()
    dice = (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return 1 - dice

def combined_loss(pred, target, bce_weight=0.5, dice_weight=0.5):
    """Combined BCE and Dice Loss
    Args:
        pred: Model predictions (sigmoid applied)
        target: Ground truth masks
        bce_weight: Weight for BCE loss
        dice_weight: Weight for Dice loss
    """
    # BCE Loss
    bce = nn.BCELoss()(pred, target)
    # Dice Loss
    dice = dice_loss(pred, target)
    # Combined loss
    return bce_weight * bce + dice_weight * dice

DICE SCORE

In [ ]:
def dice_score(pred, target):
    smooth = 1e-6
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    dice = (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return dice.item()

OPTIMIZER

In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)

PRINTING THE BATCH, CHANNEL AND SHAPE.

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)

output = model(images)

print(f"Output shape: {output.shape}")
print(f"Masks shape: {masks.shape}")

# Visualize a sample from the batch in channel 1
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Get first image from batch
sample_image = images[0, 0].cpu().numpy()  # [1, H, W] -> get first channel
sample_mask = masks[0, 0].numpy()          # [1, H, W] -> get first channel
sample_output = output[0, 0].detach().cpu().numpy()  # [1, H, W] -> get first channel

axes[0].imshow(sample_image, cmap='gray')
axes[0].set_title('Input Image (Channel 1)')
axes[0].axis('off')

axes[1].imshow(sample_mask, cmap='gray')
axes[1].set_title('Target Mask (Channel 1)')
axes[1].axis('off')

axes[2].imshow(sample_output, cmap='gray')
axes[2].set_title('Model Output (Channel 1)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

TRAIN MODEL

In [ ]:
epochs = 50
best_val_loss = float("inf")
patience = 7
counter = 0

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        predictions = model(images)

        loss = combined_loss(predictions, masks, bce_weight=0.5, dice_weight=0.5)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    val_loss = 0
    val_dice = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            predictions = model(images)
            loss = combined_loss(predictions, masks, bce_weight=0.5, dice_weight=0.5)

            val_loss += loss.item()
            val_dice += dice_score(predictions, masks)
            
    val_loss /= len(val_loader)
    val_dice /= len(val_loader)

    print(f"""
Epoch {epoch+1}/{epochs}
Train Loss: {train_loss:.4f}
Val Loss: {val_loss:.4f}
Val Dice: {val_dice:.4f}
""")

    # CHECKPOINT

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "best_resunet.pth")
        print("Model saved")
    else:
        counter += 1
        print(f"No improvement {counter}/{patience}")
    
    # EARLY STOPPING
    if counter >= patience:
        print("Early stopping")
        break

CODE TO LOAD THE MODEL

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.load_state_dict(torch.load('/content/drive/MyDrive/KidneyStone/final_model.pth',map_location=device))
model.to(device)
model.eval()
print("Model loaded successfully")

FUNCTION FOR TEST EVALUATION

In [ ]:
def dice_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()

    intersection = (pred * target).sum()
    dice = (2 * intersection + 1e-7) / (pred.sum() + target.sum() + 1e-7)

    return dice.item()


def iou_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)

    return iou.item()

THRESHOLD TO MAKE THE MODEL HAVE BINARY PREDICTION

In [ ]:
def find_best_threshold(model, val_loader):
    #Finding the threshold that maximizes Dice score
    thresholds = np.arange(0.3, 0.75, 0.05)
    best_thresh = 0.5
    best_dice = 0
    
    for thresh in thresholds:
        val_dice = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)
                preds = model(images)
                
                for i in range(images.size(0)):
                    val_dice += dice_score(preds[i], masks[i], thresh)
        
        val_dice /= len(val_loader.dataset)
        print(f"Threshold {thresh:.2f}: Dice = {val_dice:.4f}")
        
        if val_dice > best_dice:
            best_dice = val_dice
            best_thresh = thresh
    
    print(f"\n Threshold: {best_thresh:.2f} (Dice: {best_dice:.4f})")
    return best_thresh


Best_threshold = find_best_threshold(model, val_loader)

MODEL EVALUATION ON TEST DATASET

In [ ]:
model.eval()

test_loss = 0
test_dice = 0
test_iou  = 0

threshold = Best_threshold

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks  = masks.to(device)

        preds = model(images)

        test_loss += combined_loss(preds, masks).item()

        # IMPORTANT: compute per sample correctly
        for i in range(images.size(0)):
            test_dice += dice_score(preds[i], masks[i], threshold)
            test_iou  += iou_score(preds[i], masks[i], threshold)

total_samples = len(test_loader.dataset)

# Average metrics
total_samples = len(test_loader.dataset)
test_loss /= len(test_loader)
test_dice /= total_samples
test_iou /= total_samples

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice: {test_dice:.4f}")
print(f"Test IoU:  {test_iou:.4f}")


VISUALIZATION OF THE TEST DATASET WITH VOLUME

In [ ]:
model.eval()

image, mask = test_dataset[33]
with torch.no_grad():
    prediction = model(image.unsqueeze(0).to(device))

prediction = prediction.cpu().squeeze()
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(image.squeeze(), cmap="gray")
plt.title("CT")

plt.subplot(1,3,2)
plt.imshow(mask.squeeze(), cmap="gray")
plt.title("True Mask")

plt.subplot(1,3,3)
plt.imshow(prediction, cmap="gray")
plt.title("Prediction")

plt.show()


def calculate_stone_volume(mask_binary, pixel_spacing_mm=0.5, slice_thickness_mm=1.0):
    """    
    this code will calculate the volune of the predicted stone and then returns:
        volume_mm3: Volume in cubic millimeters
        volume_ml: Volume in milliliters (1 ml = 1000 mm³)
    """
    # Count pixels in the stone
    pixel_count = mask_binary.sum()
    
    # Area of one pixel in mm²
    pixel_area_mm2 = pixel_spacing_mm ** 2
    
    # Volume = area * thickness
    volume_mm3 = pixel_count * pixel_area_mm2 * slice_thickness_mm
    volume_ml = volume_mm3 / 1000  # Convert to mL
    
    return volume_mm3, volume_ml

# Example usage
pred_binary = (prediction > Best_threshold).numpy()  # Used the best threshold
volume_mm3, volume_ml = calculate_stone_volume(pred_binary)

print(f"Stone Volume: {volume_mm3:.5f} mm³")
print(f"Stone Volume: {volume_ml:.5f} mL")

VOLUME WITH OVERLAY

In [ ]:
def create_overlay_with_volume(image, mask, pred, threshold, volume_mm3):
    #Create visualization with stone overlay and volume information
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original CT
    axes[0].imshow(image.squeeze(), cmap='gray')
    axes[0].set_title('CT Image')
    axes[0].axis('off')
    
    # Overlay
    axes[1].imshow(image.squeeze(), cmap='gray')
    pred_binary = (pred > threshold).numpy()
    # Create overlay with alpha
    overlay = np.zeros((*pred_binary.shape, 4))
    overlay[pred_binary > 0] = [1, 0, 0, 0.5]  # Red with 50% opacity
    axes[1].imshow(overlay)
    axes[1].set_title(f'Stone Overlay\nVolume: {volume_mm3:.5f} mm³ ({volume_mm3/1000:.5f} mL)')
    axes[1].axis('off')
    
    # Prediction only
    axes[2].imshow(pred, cmap='gray')
    axes[2].set_title('Prediction')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Test on a sample
image, mask = test_dataset[0]
with torch.no_grad():
    pred = model(image.unsqueeze(0).to(device)).cpu().squeeze()

pred_binary = (pred > Best_threshold).numpy()
volume_mm3, volume_ml = calculate_stone_volume(pred_binary)

create_overlay_with_volume(image, mask, pred, Best_threshold, volume_mm3)